# Random Forest. Прогноз остатка

Обучаем восемь моделей на цели `(y - profile28) / 100000`. Перед оценкой возвращаем сезонный уровень и ограничиваем прогноз снизу нулём. Train содержит пять стартов с 09:30 до 11:30, проверка использует 10:30.

В `FIRST_RF_PARAMS` можно задать первую конфигурацию каждого горизонта. По умолчанию сравниваются четыре конфигурации на трёх недельных окнах. Критерий выбора равен среднему WAPE + |Relative Bias| этих окон.

После выбора модели обучаются на истории до 18 октября и оцениваются на 18-31 октября. В этом ноутбуке сохраняется оценка train/test. Финальное обучение и сабмит выполняются отдельно. Период test уже использовался в проекте.

## Запуск и результаты

В Kaggle включите NVIDIA GPU с совместимыми cuML и CuPy. Используются один GPU, `n_streams=4`, прогноз пакетами по 32 768 строк. При нехватке памяти уменьшите `PREDICTION_BATCH`.

Медианы и корреляционные лаги рассчитываются на train текущего окна. Кэш `artifacts/rf_shared_features_v2` читается с диска. Повторный **Run All** продолжает завершённые trials и оценку. В новой сессии подключите сохранённую папку `artifacts` и задайте `RESUME_FROM`. При изменении данных или параметров используйте новые `RUN_DIR` и `OUT`.

Параметры сохраняются в `model_params`, модели с предобработкой в `models/evaluation`, прогнозы в `postprocessing`, отчёты в `docs` и `reports.zip`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import gc
import os
import importlib.util
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
ALGORITHM = 'random_forest'
RESIDUAL = True
DEVICE = 'cuda' if ALGORITHM in ('xgb', 'catboost', 'random_forest') else 'cpu'
N_TRIALS = 4
N_STREAMS = 4
RF_TREES = [222, 300, 500]
RF_DEPTHS = [8, 10, 12, 16]
RF_MIN_LEAF = (4, 256)
FIRST_RF_PARAMS = {
    'n_estimators': 222,
    'max_depth': 16,
    'min_samples_leaf': 8,
    'max_features': 0.7,
    'max_samples': 0.85,
}
RESUME_FROM = None

def validate_first_params(params):
    if params is None:
        return
    bounds = {'max_features': (0.25, 0.9), 'max_samples': (0.5, 1.0)}
    if not isinstance(params, dict) or params.keys() != {'n_estimators', 'max_depth', 'min_samples_leaf', *bounds}:
        raise ValueError('FIRST_RF_PARAMS должен содержать все пять параметров леса или быть None')
    for key, choices in [('n_estimators', RF_TREES), ('max_depth', RF_DEPTHS)]:
        if type(params[key]) is not int or params[key] not in choices:
            raise ValueError(f'{key}: выберите значение из {choices} или расширьте список')
    leaf = params['min_samples_leaf']
    if type(leaf) is not int or not RF_MIN_LEAF[0] <= leaf <= RF_MIN_LEAF[1]:
        raise ValueError(f'min_samples_leaf должен быть целым в диапазоне {RF_MIN_LEAF}')
    for key, (low, high) in bounds.items():
        if type(params[key]) not in (float, int) or not low <= params[key] <= high:
            raise ValueError(f'{key}: допустимый интервал [{low}, {high}]')

validate_first_params(FIRST_RF_PARAMS)
N_JOBS = min(4, os.cpu_count() or 1)
SEED = 42
PREDICTION_BATCH = 32768
RUN_LABEL = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
required_modules = ['sklearn', 'optuna', 'joblib', 'psutil']
required_modules += [{'xgb': 'xgboost', 'catboost': 'catboost', 'lightgbm': 'lightgbm', 'prophet': 'prophet'}.get(ALGORITHM, 'sklearn')]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Установите недостающие пакеты до расчёта признаков: {sorted(set(missing))}')
if ALGORITHM == 'random_forest':
    if DEVICE != 'cuda':
        raise ValueError('Для RF выберите DEVICE=cuda')
    try:
        import cupy as cp
        from cuml.ensemble import RandomForestRegressor
        cp.zeros(1, dtype=cp.float32)
        cp.cuda.get_current_stream().synchronize()
    except (ImportError, OSError, RuntimeError) as exc:
        raise RuntimeError('Включите GPU в Kaggle и установите cuML и CuPy для CUDA этой сессии.') from exc
if N_TRIALS < 1 or PREDICTION_BATCH < 1:
    raise ValueError('Число trials и размер блока должны быть положительными')
DATA = Path('/kaggle/input/datasets/sptrent/rwb-hack')
TRAIN_FILE = 'train_solo_track.parquet'
TRAIN_PATH = DATA / TRAIN_FILE
if not TRAIN_PATH.is_file() and Path('/kaggle/input').exists():
    train_matches = list(Path('/kaggle/input').rglob(TRAIN_FILE))
    if len(train_matches) != 1:
        raise ValueError('Укажите TRAIN_PATH вручную')
    TRAIN_PATH = train_matches[0]
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
MODEL_NAME = 'random_forest_residual'
search_config = dict(first=FIRST_RF_PARAMS, trees=RF_TREES, depths=RF_DEPTHS, leaf=RF_MIN_LEAF,
                     trials=N_TRIALS, seed=SEED, streams=N_STREAMS)
search_id = hashlib.sha256(json.dumps(search_config, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR = BASE / 'artifacts' / f'{MODEL_NAME}_search_eval_{search_id}' 
OUT = BASE / 'artifacts' / 'rf_shared_features_v2'
MODEL_DIR = RUN_DIR / 'models'
POSTPROCESS_DIR = RUN_DIR / 'postprocessing'
DOCS_DIR = RUN_DIR / 'docs'
PARAMS_DIR = RUN_DIR / 'model_params'
if RESUME_FROM is not None:
    import shutil
    source = Path(RESUME_FROM)
    if not source.is_dir():
        raise ValueError('RESUME_FROM должен указывать на сохранённую папку artifacts')
    for folder_name in (RUN_DIR.name, OUT.name):
        source_folder = source / folder_name
        destination = BASE / 'artifacts' / folder_name
        if source_folder.exists() and not destination.exists():
            shutil.copytree(source_folder, destination)
for directory in (OUT, MODEL_DIR, POSTPROCESS_DIR, DOCS_DIR, PARAMS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
TARGET_SCALE = 100000
TRAIN_MINUTES = (570, 600, 630, 660, 690)
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
FEATURE_PIPELINE_VERSION = 'five_starts_windows_v4_stage_safe_hpo'

FEATURE_SOURCE_HASH = '15c604e7cc91baa57abb879c5734ad4da01c71ca9b0dea75dfdaf974d05a42f5'
MODEL_SOURCE_HASH = 'ffded4c4a204e4e1de48fff1ed126d4755489150b859517c0a710c8e25aa0857'

import importlib.metadata

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def verify_contract(path, contract):
    if path.exists():
        if json.loads(path.read_text()) != contract:
            raise ValueError(f'Настройки или данные изменились: {path}. Используйте новую папку результатов/кэша.')
    else:
        temporary = path.with_suffix('.partial')
        temporary.write_text(json.dumps(contract, ensure_ascii=False, indent=2))
        temporary.replace(path)

FEATURE_CONTRACT = dict(source=FEATURE_SOURCE_HASH, data=file_digest(TRAIN_PATH), starts=list(TRAIN_MINUTES),
                        offsets=list(WINDOW_OFFSETS), lookback=LAG_MAX_LOOKBACK_HOURS,
                        min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
verify_contract(OUT / 'contract.json', FEATURE_CONTRACT)
PREPARED = OUT / ('prepared_' + importlib.metadata.version('scikit-learn') + '_' + MODEL_SOURCE_HASH[:16])
PREPARED.mkdir(parents=True, exist_ok=True)
RUN_CONTRACT = dict(features=FEATURE_CONTRACT, source=MODEL_SOURCE_HASH, residual=RESIDUAL,
                    n_trials=N_TRIALS, seed=SEED, n_streams=N_STREAMS, trees=RF_TREES, depths=RF_DEPTHS,
                    first_params=FIRST_RF_PARAMS, min_leaf=list(RF_MIN_LEAF),
                    versions={name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'optuna')},
                    cuml=__import__('cuml').__version__)
verify_contract(RUN_DIR / 'contract.json', RUN_CONTRACT)


## Данные и временное разбиение


In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
end = times[-1]
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')
dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
selection_end = pd.Timestamp('2025-09-20')
if len(dates) < 14 or dates[-14].normalize() < selection_end or times[-1] < selection_end:
    raise ValueError('Недостаточно истории для отдельного test после подбора')
holdout_start = dates[-14]
selection_dates = dates[dates + 8 * STEP < selection_end]
if len(selection_dates) <= 28:
    raise ValueError('До 20 сентября недостаточно истории для train и трёх недель подбора')
folds = []
for start in range(len(selection_dates) - 21, len(selection_dates), 7):
    folds.append(dict(train=selection_dates[:start-7], val=selection_dates[start-7:start],
                      tuning=selection_dates[start:start+7], test=dates[-14:]))
split_dates = folds[-1]
tune_start = split_dates['tuning'][0]
LAG_VALIDATION_FIT_BEFORE = split_dates['val'][0].normalize()
display(pd.DataFrame([dict(fold=i+1, period=k, first=v[0], last=v[-1], days=len(v)) for i,f in enumerate(folds) for k,v in f.items()]))

## Общие признаки пяти стартов


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']
extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
recent = means[TARGET, 4]
usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

def offset_name(base, offset):
    if offset == 0:
        return base
    return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
    for offset in offsets:
        common[offset_name(base, offset)] = past(array, anchor + offset)

def common_features(origins):
    common = {}
    local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
    for lag in local_target_lags:
        common[f'target_back_{lag}'] = past(target, origins - lag)
    causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
    for name, array in values.items():
        for steps in (4, 12):
            add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            for lag in range(1, 5):
                common[f'{name}_back_{lag}'] = past(array, origins - lag)
            if name in STATUSES[:3]:
                change = array - past(array, positions - 2)
                add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
    for name, array in extra_signals.items():
        add_window(common, name, array, origins, causal_offsets)
    for lag in (48, 96):
        add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
    add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
    add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

for name, series in values.items():
    network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
    for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
        extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)

## Корреляционные лаги внутри train


In [ ]:
def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
    names = CORRELATION_COLUMNS + extra_columns
    result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
    if lag_table is None:
        names = [prefix + name for name in names]
        return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка не сопоставлена с историей')
        selected = lag[route]
        for j, name in enumerate(CORRELATION_COLUMNS[:4]):
            result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
        for offset in WINDOW_OFFSETS:
            source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
            available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
            signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
            available &= np.isfinite(signal)
            result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
            result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
    names = [prefix + name for name in names]
    frame = frame.drop(columns=names, errors='ignore')
    return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
def remove_seasonality(series):
    typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
    return series - typical

residual_target = remove_seasonality(target)
residual_signal = remove_seasonality(hub_signal)

def fit_lags(boundary):
    tables = []
    for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
        table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
        tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
    return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

def add_correlation_features(frame,h,lag_table):
    frame = correlation_block(frame,h,lag_table,hub_signal,'')
    if lag_table is None:
        return correlation_block(frame,h,None,residual_signal,'residual_')
    residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
    residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
    return correlation_block(frame,h,residual_table,residual_signal,'residual_')

## Общие матрицы восьми горизонтов


In [ ]:
def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    common = common_features(origins)
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
    calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, None)
    seasonal_columns = {}
    for offset in WINDOW_OFFSETS:
        for day in (1, 2, 7):
            seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
        for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
            seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
    for base in ('slot_daily7','slot_weekly4'):
        window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
        valid = np.isfinite(window)
        maximum = np.where(valid, window, -np.inf).max(axis=1)
        minimum = np.where(valid, window, np.inf).min(axis=1)
        peak = valid & (window == maximum[:,None])
        count = peak.sum(axis=1)
        shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
        seasonal_columns[base+'_peak_minutes'] = shift * 30
        seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
        seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
    frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    raw_target_sources = []
    for lag in range(9):
        raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
    for lag in (48, 96):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
    for day in (1, 2, 7):
        for offset in WINDOW_OFFSETS:
            raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
    seen, redundant = (set(), [])
    for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
        if source in seen:
            redundant.append(name)
        else:
            seen.add(source)
    frame = frame.drop(columns=redundant)
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric}).copy()

def write_horizon(h):
    minutes = times.hour * 60 + times.minute
    origins = np.flatnonzero(np.isin(minutes, TRAIN_MINUTES) & (np.arange(len(times)) + h < len(times)))
    path = OUT / f'history_h{h}.parquet'
    if path.exists():
        print(f'h={h}: общие признаки загружены из кэша')
        return
    temporary = path.with_suffix('.partial')
    writer = None
    try:
        for start in range(0, len(origins), 35):
            batch = origins[start:start+35]
            frame = horizon_dataset(h, batch)
            frame['y'] = target[batch+h].ravel()
            table = pa.Table.from_pandas(frame,preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temporary,table.schema,compression='zstd')
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(path)
    print(f'h={h}: история {len(origins)*len(routes):,}, модельных признаков {len(frame.columns) - 5}')
for h in range(1, 9):
    write_horizon(h)
feature_names = list(dict.fromkeys((name for h in range(1, 9) for name in pq.read_schema(OUT / f'history_h{h}.parquet').names)))
FEATURE_GROUPS = {'calendar': ['dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'))) and 'lag_48' not in name], 'hub_combinations': [name for name in feature_names if any((name.startswith(prefix) for prefix in ('status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'))) and 'lag_48' not in name], 'historical_lags': [name for name in feature_names if 'lag_48' in name or 'lag_96' in name], 'target_change': [name for name in feature_names if name.startswith('target_acceleration')], 'correlation_lag': [name for name in feature_names if name in CORRELATION_COLUMNS or name.startswith('lagged_status6_signal')], 'seasonal_windows': [name for name in feature_names if name.startswith('slot_')], 'recent_target': [name for name in feature_names if name.startswith('target_back_')], 'recent_statuses': [name for name in feature_names if any((name.startswith(f'status_{i}_back_') or name == f'status_{i}_latest' for i in range(1, 7)))]}

del values, means, level24, extra_signals, positions, recent, usual
FEATURE_GROUPS['network'] = [name for name in feature_names if name.startswith('network_')]
FEATURE_GROUPS['residual_correlation_lag'] = [name for name in feature_names if name.startswith('residual_')]
(DOCS_DIR / 'feature_pipeline.json').write_text(json.dumps({'version': FEATURE_PIPELINE_VERSION, 'window_offsets_steps': list(WINDOW_OFFSETS), 'training_origins_minutes': TRAIN_MINUTES, 'evaluation_origin': '10:30', 'correlation_storage': 'NaN placeholders; fit on stage train before model use', 'validation_lag_fit_before': str(LAG_VALIDATION_FIT_BEFORE), 'selection_end_exclusive': str(selection_end), 'all_folds': [{key: [str(period.min()), str(period.max())] for key, period in fold.items()} for fold in folds], 'splits': {key: [str(period.min()), str(period.max())] for key, period in split_dates.items()}, 'feature_groups': FEATURE_GROUPS}, ensure_ascii=False, indent=2))

## Метрика


In [ ]:
def mean_daily_score(frame, prediction):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(prediction, dtype=np.float64) - actual
    totals = pd.DataFrame({'cutoff': frame.cutoff.dt.normalize().to_numpy(), 'volume': actual, 'error': errors, 'absolute': np.abs(errors)}).groupby('cutoff').sum()
    denominator = totals.volume.where(totals.volume > 0)
    return ((totals.absolute + totals.error.abs()) / denominator).mean()

def score_row(frame, predictions, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    errors = np.asarray(predictions, dtype=np.float64) - actual
    volume = actual.sum()
    wape = np.abs(errors).sum() / volume if volume > 0 else np.nan
    signed_bias = errors.sum() / volume if volume > 0 else np.nan
    return {'выборка': split, 'даты': frame.cutoff.dt.normalize().nunique(), 'строки': len(frame), 'WAPE': wape, 'Relative Bias': signed_bias, '|Relative Bias|': abs(signed_bias), 'метрика конкурса': wape + abs(signed_bias), 'среднее по дням': mean_daily_score(frame, predictions)}
TARGET_SCALE = 100000

def model_columns(data):
    return data.columns.drop(['timestamp', 'cutoff', 'y', 'profile28', 'overlap']).tolist()

## Кэширование, модели и продолжение поиска


In [ ]:
import time
import joblib
import optuna
import cuml
import sklearn
import zipfile
from sklearn.impute import SimpleImputer


def atomic_json(path, value):
    temporary = path.with_suffix('.partial')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2))
    temporary.replace(path)


def model_target(frame):
    baseline = frame.profile28.to_numpy() if RESIDUAL else 0
    return ((frame.y.to_numpy() - baseline) / TARGET_SCALE).astype(np.float32)


def transformed_frame(frame, columns):
    result = frame[columns].drop(columns='route_id').astype(np.float32)
    codes = pd.Index(routes).get_indexer(frame.route_id).astype(np.int64) + 1
    bits = {f'route_bit_{i}': ((codes >> i) & 1).astype(np.float32)
            for i in range(max(1, int(len(routes)).bit_length()))}
    return pd.concat([result, pd.DataFrame(bits, index=frame.index)], axis=1).copy()


def cached_lags(boundary):
    path = OUT / f'lags_{pd.Timestamp(boundary).strftime("%Y%m%dT%H%M%S")}.parquet'
    if path.exists():
        return pd.read_parquet(path)
    table = fit_lags(boundary)
    temporary = path.with_suffix('.partial')
    table.to_parquet(temporary, index=False)
    temporary.replace(path)
    return table


def prepare_stage(h, stage, train_dates=None, validation_dates=None, boundary=None):
    folder = PREPARED / f'h{h}_{stage}'
    marker = folder / 'ready.json'
    if marker.exists():
        return folder
    folder.mkdir(parents=True, exist_ok=True)
    filters = [('timestamp', '<', selection_end)] if stage.startswith('fold') else None
    history = pd.read_parquet(OUT / f'history_h{h}.parquet', filters=filters)
    history = history.loc[np.isfinite(history.profile28)].copy()
    mask = history.timestamp < boundary
    if train_dates is not None:
        mask &= history.cutoff.dt.normalize().isin(train_dates.normalize())
    train = history.loc[mask].copy()
    valid = history.loc[history.cutoff.isin(validation_dates)].copy()
    del history
    if train.empty or valid.empty or train.timestamp.max() >= valid.cutoff.min().normalize():
        raise ValueError('Нарушено временное разделение')
    if stage.startswith('fold') and valid.timestamp.max() >= selection_end:
        raise ValueError('Tuning выходит за границу подбора')
    table = cached_lags(boundary)
    train = add_correlation_features(train, h, table)
    valid = add_correlation_features(valid, h, table)
    columns = model_columns(train)
    prep = SimpleImputer(strategy='median', keep_empty_features=True)
    for name, frame in [('train', train), ('valid', valid)]:
        encoded = transformed_frame(frame, columns)
        matrix = prep.fit_transform(encoded) if name == 'train' else prep.transform(encoded)
        np.save(folder / f'{name}.npy', np.asarray(matrix, dtype=np.float32))
        frame[['cutoff', 'timestamp', 'route_id', 'y', 'profile28']].to_parquet(folder / f'{name}.parquet', index=False)
        del matrix, encoded
    joblib.dump(prep, folder / 'preprocess.joblib')
    atomic_json(marker, dict(columns=columns, boundary=str(boundary), rows_train=len(train), rows_valid=len(valid)))
    return folder


def load_stage(folder):
    return {name: (pd.read_parquet(folder / f'{name}.parquet'), np.load(folder / f'{name}.npy', mmap_mode='r')) for name in ('train', 'valid')}


def fit_forest(frame, matrix, params):
    model = RandomForestRegressor(random_state=SEED, n_streams=N_STREAMS, output_type='numpy', **params)
    model.fit(matrix, model_target(frame))
    cp.cuda.get_current_stream().synchronize()
    return model


def predict_matrix(model, frame, matrix):
    result = np.empty(len(frame), dtype=np.float64)
    for start in range(0, len(frame), PREDICTION_BATCH):
        stop = min(start + PREDICTION_BATCH, len(frame))
        raw = np.asarray(model.predict(matrix[start:stop]), dtype=np.float64).reshape(-1)
        baseline = frame.profile28.iloc[start:stop].to_numpy() if RESIDUAL else 0
        result[start:stop] = np.maximum(0, raw * TARGET_SCALE + baseline)
    if not np.isfinite(result).all():
        raise ValueError('Некорректный прогноз')
    return result


def prediction_part(frame, prediction, h, period):
    part = frame.copy()
    part['prediction'] = prediction
    part[ALGORITHM] = prediction
    part['horizon'] = h
    part['period'] = period
    return part


def report(rows, phase, h, trial=None, fold=None, seconds=None):
    records = [dict(phase=phase, horizon=h, trial=trial, fold=fold, seconds=seconds, **row) for row in rows]
    path = DOCS_DIR / f'{phase}_h{h}_trial{trial}_fold{fold}.json'
    atomic_json(path, records)
    display(pd.DataFrame(records))


def suggest_params(trial):
    return dict(n_estimators=trial.suggest_categorical('n_estimators', RF_TREES),
                max_depth=trial.suggest_categorical('max_depth', RF_DEPTHS),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', *RF_MIN_LEAF, log=True),
                max_features=trial.suggest_float('max_features', 0.25, 0.9),
                max_samples=trial.suggest_float('max_samples', 0.5, 1))


def select_horizon(h):
    sampler_path = POSTPROCESS_DIR / f'sampler_h{h}.joblib'
    sampler = joblib.load(sampler_path) if sampler_path.exists() else optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction='minimize', sampler=sampler,
                               storage=f'sqlite:///{(POSTPROCESS_DIR / "search.sqlite3").resolve().as_posix()}',
                               study_name=f'h{h}', load_if_exists=True)
    if FIRST_RF_PARAMS is not None and not study.trials:
        study.enqueue_trial(FIRST_RF_PARAMS)
    for trial in study.get_trials(states=(optuna.trial.TrialState.RUNNING,)):
        study.tell(trial.number, state=optuna.trial.TrialState.FAIL)
        retry_params = {**trial.system_attrs.get('fixed_params', {}), **trial.params}
        if retry_params:
            study.enqueue_trial(retry_params)
    if FIRST_RF_PARAMS is not None and not any(
        t.state in (optuna.trial.TrialState.WAITING, optuna.trial.TrialState.COMPLETE) for t in study.trials
    ):
        study.enqueue_trial(FIRST_RF_PARAMS)
    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
    remaining = max(0, N_TRIALS - len(completed))
    folders = []
    if remaining:
        for i, fold in enumerate(folds):
            folders.append(prepare_stage(h, f'fold{i+1}', fold['train'], fold['tuning'], fold['val'][0].normalize()))
    def objective(trial):
        started = time.perf_counter()
        params = suggest_params(trial)
        print(f"h={h}, trial={trial.number}, параметры: {params}")
        scores, parts = [], []
        for i, folder in enumerate(folders):
            fold_started = time.perf_counter()
            data = load_stage(folder)
            train, x_train = data['train']
            tuning, x_tuning = data['valid']
            model = fit_forest(train, x_train, params)
            prediction = predict_matrix(model, tuning, x_tuning)
            rows = [score_row(train, predict_matrix(model, train, x_train), 'train'), score_row(tuning, prediction, 'tuning')]
            scores.append(rows[-1]['метрика конкурса'])
            part = prediction_part(tuning, prediction, h, 'tuning')
            part['fold'] = i + 1
            part['fit_labels_before'] = folds[i]['val'][0].normalize()
            part['selection_labels_through'] = folds[-1]['tuning'][-1] + h * STEP
            part['model_info_through'] = part['selection_labels_through']
            part['is_unbiased_oof'] = False
            parts.append(part)
            report(rows, 'search', h, trial.number, i + 1, time.perf_counter() - fold_started)
            del model, data, train, tuning, x_train, x_tuning
            gc.collect()
        value = float(np.mean(scores))
        if not np.isfinite(value):
            raise ValueError('Неконечная метрика подбора')
        path = POSTPROCESS_DIR / f'tuning_h{h}_trial{trial.number}.parquet'
        temporary = path.with_suffix('.partial')
        pd.concat(parts, ignore_index=True).to_parquet(temporary, index=False)
        temporary.replace(path)
        trial.set_user_attr('fold_scores', scores)
        trial.set_user_attr('elapsed_seconds', time.perf_counter() - started)
        return value
    def save_sampler(study, trial):
        temporary = sampler_path.with_suffix('.partial')
        joblib.dump(study.sampler, temporary)
        temporary.replace(sampler_path)
    if remaining:
        study.optimize(objective, n_trials=remaining, n_jobs=1, gc_after_trial=True, callbacks=[save_sampler])
    study.trials_dataframe().to_csv(DOCS_DIR / f'trials_h{h}.csv', index=False)
    best = study.best_trial
    best_part = pd.read_parquet(POSTPROCESS_DIR / f'tuning_h{h}_trial{best.number}.parquet')
    best_part.to_parquet(POSTPROCESS_DIR / f'tuning_predictions_h{h}.parquet', index=False)
    chosen = dict(algorithm=ALGORITHM, residual=RESIDUAL, horizon=h, params=best.params, trial=best.number,
                  tuning_score=best.value, fold_scores=best.user_attrs['fold_scores'], trees=best.params['n_estimators'],
                  selection_end_exclusive=str(selection_end), selection_labels_through=str(folds[-1]['tuning'][-1] + h * STEP))
    atomic_json(PARAMS_DIR / f'best_h{h}.json', chosen)
    durations = [t.user_attrs['elapsed_seconds'] for t in study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))]
    print(f'h={h}: завершено {len(durations)} trials, среднее {np.mean(durations)/60:.1f} мин/trial')
    return chosen


manifest = dict(algorithm=ALGORITHM, residual=RESIDUAL, seed=SEED, device=DEVICE, n_trials=N_TRIALS,
                n_streams=N_STREAMS, prediction_batch=PREDICTION_BATCH, search_trees=RF_TREES, search_depths=RF_DEPTHS,
                feature_cache=str(OUT), prepared_cache=str(PREPARED), holdout_start=str(holdout_start),
                selection_end_exclusive=str(selection_end), contract=RUN_CONTRACT,
                test_note='Независимая оценка test получена на моделях, обученных только на истории до test, включая train и val. Переобучение на всей истории и submission не выполняются. Период test ранее использовался в проекте; это не новый слепой holdout.')
atomic_json(DOCS_DIR / 'manifest.json', manifest)

## Подбор параметров до test


In [ ]:
selected = {}
for h in range(1, 9):
    selected[h] = select_horizon(h)
    gc.collect()
atomic_json(PARAMS_DIR / 'selection_frozen.json', selected)

## Обучение на train + val и независимая оценка test


In [ ]:
evaluation_rows = []
for h in range(1, 9):
    checkpoint = MODEL_DIR / 'evaluation' / f'h{h}'
    done = checkpoint / 'complete.json'
    if done.exists():
        rows = json.loads(done.read_text())['rows']
        display(pd.DataFrame(rows))
    else:
        started = time.perf_counter()
        folder = prepare_stage(h, 'evaluation', validation_dates=folds[-1]['test'], boundary=holdout_start.normalize())
        data = load_stage(folder)
        train, x_train = data['train']
        model = fit_forest(train, x_train, selected[h]['params'])
        rows = []
        for name, split in [('train', 'train'), ('valid', 'test')]:
            frame, matrix = data[name]
            prediction = predict_matrix(model, frame, matrix)
            rows.append(score_row(frame, prediction, split))
            part = prediction_part(frame, prediction, h, split)
            part['model_info_through'] = holdout_start.normalize() - STEP
            part['is_unbiased_oof'] = split == 'test'
            path = POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet'
            temporary = path.with_suffix('.partial')
            part.to_parquet(temporary, index=False)
            temporary.replace(path)
        checkpoint.mkdir(parents=True, exist_ok=True)
        metadata = json.loads((folder / 'ready.json').read_text())
        metadata.update(algorithm=ALGORITHM, residual=RESIDUAL, target_scale=TARGET_SCALE, route_categories=routes.tolist(), route_encoding='binary_index_plus_one', params=selected[h]['params'])
        bundle = dict(model=model, preprocess=joblib.load(folder / 'preprocess.joblib'), metadata=metadata)
        temporary = checkpoint / 'model.partial'
        joblib.dump(bundle, temporary, compress=0)
        temporary.replace(checkpoint / 'model.joblib')
        atomic_json(checkpoint / 'metadata.json', metadata)
        report(rows, 'independent_evaluation', h, seconds=time.perf_counter() - started)
        atomic_json(done, dict(rows=rows))
        del bundle, model, data, train, x_train, frame, matrix, part
        gc.collect()
    evaluation_rows.extend([dict(horizon=h, **row) for row in rows])
pd.DataFrame(evaluation_rows).to_csv(DOCS_DIR / 'independent_metrics_by_horizon.csv', index=False)
aggregate_rows = []
for split in ('train', 'test'):
    part = pd.concat([pd.read_parquet(POSTPROCESS_DIR / f'{split}_predictions_h{h}.parquet') for h in range(1, 9)], ignore_index=True)
    aggregate_rows.append(score_row(part, part.prediction, split))
    aggregate_rows.append(score_row(part, part.profile28, split + ' profile28'))
    if split == 'test':
        part.to_parquet(POSTPROCESS_DIR / 'test_predictions.parquet', index=False)
    del part
independent_metrics = pd.DataFrame(aggregate_rows)
display(independent_metrics)
independent_metrics.to_csv(DOCS_DIR / 'independent_metrics.csv', index=False)
(DOCS_DIR / 'evaluation_note.txt').write_text(manifest['test_note'])
records = []
for path in sorted(DOCS_DIR.glob('*_h*_trial*_fold*.json')):
    records.extend(json.loads(path.read_text()))
pd.DataFrame(records).to_csv(DOCS_DIR / 'metrics.csv', index=False)
with zipfile.ZipFile(RUN_DIR / 'reports.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (DOCS_DIR, PARAMS_DIR):
        for path in folder.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(RUN_DIR))
print(manifest['test_note'])
print(f'Результаты: {RUN_DIR.resolve()}')

In [ ]:
import shutil
from pathlib import Path

base = RUN_DIR

for folder in ["docs", "model_params"]:
    shutil.make_archive(
        str(BASE / folder),
        "zip",
        root_dir=base,
        base_dir=folder
    )
    print(f"{folder}.zip создан")
